# Andina Market Capa Gold (Star Schema)

Construye el modelo dimensional a partir de `lh_andina_market.silver.*` y lo
deja listo para el modelo semantico de Power BI (Direct Lake).

**Decisiones de diseño:**
- Modelo simplificado usando **claves naturales** (`CustomerID`, `ProductID`, fechas directas) en lugar de surrogate keys.
- Grano de cada fact: `fact_orders` = 1 fila por pedido, `fact_order_items` = 1 fila por línea de pedido, `fact_payments` = 1 fila por pago, `fact_support_tickets` = 1 fila por ticket.


## 1. dim_date

Calendario generado desde el rango real de fechas de negocio (con 7 dias de margen a cada lado), no hardcodeado.

In [0]:
from pyspark import pipelines as dp

@dp.materialized_view(name="andina_market.gold.dim_date")
def dim_date():
    return spark.sql("""
        WITH bounds AS (
            SELECT
                LEAST(MIN(OrderDate), MIN(CreatedAt))    AS min_date,
                GREATEST(MAX(OrderDate), MAX(CreatedAt)) AS max_date
            FROM andina_market.silver.orders
        ),
        calendar AS (
            SELECT explode(sequence(
                to_date(min_date) - INTERVAL 7 DAY,
                to_date(max_date) + INTERVAL 7 DAY,
                INTERVAL 1 DAY
            )) AS FullDate
            FROM bounds
        )
        SELECT
            FullDate,
            YEAR(FullDate)                                       AS Year,
            QUARTER(FullDate)                                    AS Quarter,
            MONTH(FullDate)                                       AS Month,
            date_format(FullDate, 'MMMM')                         AS MonthName,
            DAY(FullDate)                                         AS Day,
            date_format(FullDate, 'EEEE')                         AS DayName,
            CASE WHEN DAYOFWEEK(FullDate) IN (1, 7) THEN TRUE ELSE FALSE END AS IsWeekend
        FROM calendar
    """)

## 2. dim_product (SCD Tipo 1)

Usa `ProductID` como clave natural. El notebook hace *full refresh* de toda la capa gold en cada ejecución (bronze/silver también se recalculan completos).

In [0]:
@dp.materialized_view(name="andina_market.gold.dim_product")
def dim_product():
    return spark.sql("""
        SELECT
            ProductID,
            SKU,
            ProductName,
            Category,
            UnitPrice AS CurrentUnitPrice,
            Status,
            had_invalid_price
        FROM andina_market.silver.products
    """)

## 3. dim_customer

In [0]:
@dp.materialized_view(name="andina_market.gold.dim_customer")
def dim_customer():
    return spark.sql("""
        SELECT
            CustomerID,
            FullName,
            Email,
            Phone,
            City,
            Country,
            Segment,
            SignupDate,
            CURRENT_DATE() AS EffectiveDate,
            CAST(NULL AS DATE) AS ExpirationDate,
            TRUE AS IsCurrent
        FROM andina_market.silver.customers
    """)

## 4. fact_orders

Grano: 1 fila por pedido. Usa `CustomerID` y `OrderDate` directamente sin surrogate keys.

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_orders")
def fact_orders():
    return spark.sql("""
        SELECT
            o.OrderID,
            o.CustomerID,
            o.OrderDate,
            o.Channel,
            o.Status,
            o.TotalAmount
        FROM andina_market.silver.orders o
    """)

## 5. fact_order_items

Grano: 1 fila por linea de pedido.

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_order_items")
def fact_order_items():
    return spark.sql("""
        SELECT
            oi.OrderItemID,
            oi.OrderID,
            oi.ProductID,
            oi.Quantity,
            oi.UnitPrice,
            ROUND(oi.Quantity * oi.UnitPrice, 2) AS LineTotal
        FROM andina_market.silver.order_items oi
    """)

## 6. fact_payments

Grano: 1 fila por pago (incluye reintentos, ya numerados en silver con `payment_attempt_number`).

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_payments")
def fact_payments():
    return spark.sql("""
        SELECT
            p.PaymentID,
            p.OrderID,
            p.PaymentDate,
            p.PaymentMethod,
            p.Amount,
            p.Status,
            p.payment_attempt_number,
            p.amount_mismatch_flag
        FROM andina_market.silver.payments p
    """)

## 7. fact_support_tickets

Grano: 1 fila por ticket. Incluye `DaysToUpdate` como metrica base para un KPI de tiempo de resolucion en el Nivel 3.

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_support_tickets")
def fact_support_tickets():
    return spark.sql("""
        SELECT
            t.TicketID,
            t.CustomerID,
            t.CreatedAt,
            t.Status,
            t.Priority,
            DATEDIFF(t.UpdatedAt, t.CreatedAt) AS DaysToUpdate
        FROM andina_market.silver.support_tickets t
    """)